In [8]:
%pip install openai python-dotenv groq

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()


@dataclass(frozen=True)
class Provider:
    """One provider described as pure DATA (same design as Notebook 1)."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider("Groq",       "GROQ_API_KEY",       True,  "https://api.groq.com/openai/v1", "openai/gpt-oss-20b"),
    Provider("OpenAI",     "OPENAI_API_KEY",     False, None,                              "gpt-4o-mini"),
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return provider
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set. Add one of {expected} to your .env file.")


def build_client(provider: Provider):
    from openai import OpenAI

    api_key = os.environ[provider.env_var]
    if provider.base_url is None:
        return OpenAI(api_key=api_key)
    return OpenAI(api_key=api_key, base_url=provider.base_url)


def have_any_key() -> bool:
    return any(os.environ.get(p.env_var) for p in PROVIDERS)



def llm_reply(prompt: str, *, max_tokens: int = 400) -> str:
    """Send one user prompt; return the assistant's text."""
    provider = select_provider()
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return result.choices[0].message.content

# Zero Shot COT

In [14]:

ZERO_SHOT_COT_SUFFIX = "\n\nLet's think step by step."

def zero_shot_cot_prompt(question: str) -> str:
    """This function takes a question and returns a prompt for the zero-shot COT model."""
    return f"Question: {question}{ZERO_SHOT_COT_SUFFIX}"

In [15]:
QUESTION = (
    "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
    "Each can has 3 tennis balls. How many tennis balls does he have now?"
)

In [16]:
print(zero_shot_cot_prompt(QUESTION))

Question: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?

Let's think step by step.


In [17]:
reply = llm_reply(zero_shot_cot_prompt(QUESTION))

print(reply)

**Step 1: Identify the initial amount**  
Roger starts with **5 tennis balls**.

**Step 2: Calculate the number of balls in the new cans**  
- He buys **2 cans**.  
- Each can contains **3 tennis balls**.  

So the total from the cans is:
\[
2 \text{ cans} \times 3 \text{ balls per can} = 6 \text{ balls}
\]

**Step 3: Add the new balls to the initial amount**  
\[
5 \text{ (original balls)} + 6 \text{ (new balls)} = 11 \text{ balls}
\]

**Answer:** Roger now has **11 tennis balls**.


# Few shot COT

In [18]:
FEW_SHOT_EXAMPLES = """
Q: There are 15 trees in the grove. Grove workers will plant 6 trees today. How many trees will be in the grove?
A: There are 15 trees to start. After planting 6 more, there are 15 + 6 = 21 trees.
Final answer: 21

Q: Leah had 32 chocolates and her sister had 42. They ate 35. How many pieces do they have left in total?
A: Leah and her sister had 32 + 42 = 74 chocolates together. After eating 35, they have 74 - 35 = 39 left.
Final answer: 39
""".strip()

In [19]:
def few_shot_cot_prompt(question: str) -> str:
    """This function takes a question and returns a prompt for the few-shot COT model."""
    return f"""Solve each question by reasoning step by step.
    {FEW_SHOT_EXAMPLES}
    
    Question: {question}
    Answer:
    """

In [20]:
reply = llm_reply(few_shot_cot_prompt(QUESTION))

print(reply)

A: Roger starts with 5 tennis balls.  
He buys 2 cans, and each can contains 3 tennis balls, so the number of new balls is  
\(2 \times 3 = 6\).  
Adding these to the original 5 gives  
\(5 + 6 = 11\) tennis balls.

Final answer: **11**


In [21]:
TRICKY = (
    "A juggler can juggle 16 balls. Half of the balls are golf balls, and "
    "half of the golf balls are blue. How many blue golf balls are there?"
)

In [22]:
reply = llm_reply(zero_shot_cot_prompt(TRICKY))

print(reply)

**Step 1: Identify the total number of golf balls.**  
The juggler has 16 balls in all.  
Half of those are golf balls:  

\[
\frac{1}{2}\times 16 = 8 \text{ golf balls.}
\]

**Step 2: Find how many of those golf balls are blue.**  
Half of the golf balls are blue:  

\[
\frac{1}{2}\times 8 = 4 \text{ blue golf balls.}
\]

---

\[
\boxed{4}
\]


In [23]:
reply = llm_reply(few_shot_cot_prompt(TRICKY))

print(reply)

The juggler has a total of 16 balls.  
- Half of them are golf balls:  
  \( \frac{1}{2} \times 16 = 8 \) golf balls.  
- Half of those golf balls are blue:  
  \( \frac{1}{2} \times 8 = 4 \) blue golf balls.

**Answer:** 4 blue golf balls.
